In [ ]:
import numpy as np
from pathlib import Path
import open3d as o3d
import matplotlib.pyplot as plt
import igl
import plotly.graph_objects as go
import plotly.express as px
from scipy.sparse.linalg import spsolve
import networkx as nx
from fold_utils import plane_mesh_slice, plane_mesh_slices_single_normal, get_triangle_adjacency_matrix, get_segment_path_from_triangle_path
from scipy.spatial.transform import Rotation
from scipy.fft import dst, idst
from fold_utils import arclength_resample, curvature_dst
from scipy.interpolate import CubicSpline

In [ ]:
def solve_dirichlet(A, b, bd_indices=None, val_at_bd=None):
    n = A.shape[0]

    if bd_indices is None or len(bd_indices) == 0:
        return spsolve(A, b)

    bd_indices = np.asarray(bd_indices)

    if val_at_bd is None:
        val_at_bd = np.zeros(len(bd_indices))
    else:
        val_at_bd = np.asarray(val_at_bd)
        if val_at_bd.shape[0] == n:
            val_at_bd = val_at_bd[bd_indices]

    free = np.setdiff1d(np.arange(n), bd_indices)

    u = np.zeros(n)
    u[bd_indices] = val_at_bd

    rhs = b[free] - A[free][:, bd_indices] @ u[bd_indices]
    u[free] = spsolve(A[free][:, free], rhs)

    return u

In [ ]:
source_folder = 'local_data/cut_meshes/'
files = list(Path(source_folder).glob('*.ply'))
for file in files:
    if 'sub' in source_folder:
        if not file.name.endswith('dome_submesh.ply'):
            continue
    mesh = o3d.io.read_triangle_mesh(file)
    mesh.compute_vertex_normals()
    mesh.compute_adjacency_list()
    
    adj_graph = nx.from_dict_of_lists( {idx: neighbors for idx, neighbors in enumerate(mesh.adjacency_list)})
    
    vertices, triangles = np.asarray(mesh.vertices), np.asarray(mesh.triangles)
    triangle_centers = vertices[triangles].mean(axis=1)
    tri_adj_matrix = get_triangle_adjacency_matrix(triangles)
    tri_adj_graph = nx.from_scipy_sparse_array(tri_adj_matrix)
    edges = np.concatenate([triangles[:,[0,1]], triangles[:,[1,2]], triangles[:,[2,0]]])
    edges = np.unique(np.sort(edges, axis=1), axis=0)
    edge_lengths = np.linalg.norm(vertices[edges[:, 0]] - vertices[edges[:, 1]], axis=1)
    # add weighted edges
    for (i, j), length in zip(edges, edge_lengths):
        adj_graph.add_edge(i, j, weight=length)
    
    vertex_normals = np.asarray(mesh.vertex_normals)
    boundary_loop = igl.boundary_loop(triangles)
    boundary_edges = igl.boundary_facets(triangles)
    boundary_vertices = np.unique(boundary_edges[0].flatten())
    principal_curvatures = igl.principal_curvature(vertices, triangles)
    vertex_pc1_directions, vertex_pc2_directions = principal_curvatures[0], principal_curvatures[1]
    vertex_pc1_values, vertex_pc2_values = principal_curvatures[2], principal_curvatures[3]

    # cotangent Laplacian
    L = igl.cotmatrix(vertices, triangles)          # usually negative semi-definite
    M = igl.massmatrix(vertices, triangles, igl.MASSMATRIX_TYPE_VORONOI)

    
    field = solve_dirichlet(-L, M @ np.ones(len(vertices))/len(vertices), bd_indices=boundary_vertices, val_at_bd=np.zeros(len(boundary_vertices)))
    field_threshold = np.quantile(field, 0.9)
    region_indices = np.where(field > field_threshold)[0]
    rho = np.zeros(len(vertices))
    rho[region_indices] = 1.0 / len(region_indices)
    
    alpha = 1e-2  # tune this
    
    A = M - alpha * L
    b = M @ rho
    
    weights = solve_dirichlet(A, b)   # no boundary constraints
    centroid_idx = region_indices[np.argmax(field[region_indices])]
    centroid_tris =  np.argwhere(np.any(triangles == centroid_idx, axis=1)).flatten()
    centroid = vertices[centroid_idx]
    centroid_normal = vertex_normals[centroid_idx]

    
    



    #bd_pos = vertices[boundary_loop]    
    ## edge-length parametrization along boundary
    #edges = np.linalg.norm(np.roll(bd_pos, -1, axis=0) - bd_pos, axis=1)
    #s = np.concatenate([[0.0], np.cumsum(edges[:-1])])
    #theta = 2 * np.pi * s / edges.sum()
#
    #u_bd = np.cos(theta)
    #v_bd = np.sin(theta)
#
#
    #full_boundary_indices = np.concatenate([boundary_loop, np.array([centroid_idx])])
    #full_boundary_u = np.concatenate([u_bd, [0.0]])
    #full_boundary_v = np.concatenate([v_bd, [0.0]])
#
    #u = solve_dirichlet(-L, np.zeros(len(vertices)), full_boundary_indices, full_boundary_u)
    #v = solve_dirichlet(-L, np.zeros(len(vertices)), full_boundary_indices, full_boundary_v)
#
    #UV = np.column_stack([u, v])



    #all_paths = nx.single_source_dijkstra(adj_graph, centroid_idx, weight='weight')
    #distances = np.array([all_paths[0].get(i, float('inf')) for i in range(len(vertices))])
    #avg_normal = np.zeros(3)
    #centers = []
    #dr = 5
    #for i in range(1, 15):
    #    mask = (distances < dr * (i + 1)) * (distances >= dr * i)
    #    if mask.sum() <= 3:
    #        continue
    #    masked_vertices = vertices[mask]
    #    masked_vertices_centroid = masked_vertices.mean(axis=0)
    #    mv_vals, mv_vecs = np.linalg.eigh(np.cov(masked_vertices, rowvar=False))
    #    mv_normal = mv_vecs[:, 0]
    #    avg_normal += mv_normal
    #    centers.append(masked_vertices_centroid)
    #avg_normal /= np.linalg.norm(avg_normal)
    #centers = np.array(centers)
    #all_segments,plane_ids, all_tri_ids, _ = plane_mesh_slices_single_normal(vertices, triangles, centers, avg_normal, dr)
    #
    #for i, center in enumerate(centers):
    #    plane_mask = plane_ids == i
    #    segments = all_segments[plane_mask]
    #    tri_ids = all_tri_ids[plane_mask]
    #    seg_index_of = {tri: i for i, tri in enumerate(tri_ids)}
    #    if len(segments) == 0:
    #        continue
    #    for group in nx.connected_components(tri_adj_graph.subgraph(tri_ids)):
    #        _, seg_indices = get_segment_path_from_triangle_path(tri_adj_graph, list(group), seg_index_of)
    #        lp = (segments[seg_indices][:, 0] + segments[seg_indices][:, 1]) / 2


    

    fig = go.Figure()
    trace = go.Mesh3d(
        x=vertices[:, 0],
        y=vertices[:, 1],
        z=vertices[:, 2],
        i=triangles[:, 0],
        j=triangles[:, 1],
        k=triangles[:, 2],
        intensity=field,
        colorscale='Viridis',
        showscale=True,
        opacity=0.8
    )
    fig.add_trace(trace)
    fig.add_trace(go.Scatter3d(
        x=[centroid[0]],
        y=[centroid[1]],
        z=[centroid[2]],
        mode='markers',
        marker=dict(color='red', size=5)
    ))
    fig.add_trace(go.Scatter3d(
        x=vertices[boundary_vertices][:, 0],
        y=vertices[boundary_vertices][:, 1],
        z=vertices[boundary_vertices][:, 2],
        mode='markers',
        marker=dict(color='black', size=3)
    ))
    fig.update_layout(title=f'{file.name}', scene=dict(aspectmode='data'))
    

    num_angles = 50
    dtheta = (np.pi) / num_angles
    plane_normals = [vertex_pc1_directions[centroid_idx] - vertex_pc1_directions[centroid_idx].dot(centroid_normal) * centroid_normal]
    for adx in range(1, num_angles):
        plane_normals.append(Rotation.from_rotvec(centroid_normal * dtheta).apply(plane_normals[adx-1]))
    plane_normals = np.array(plane_normals)
    #_, axes = plt.subplots(nrows=3, ncols=int(np.ceil(len(plane_normals)/3).item()), figsize=(30, 10))
    #axes = axes.flatten()
    ax = plt.subplot(1,1,1)
    for plane_idx, plane_normal in enumerate(plane_normals):
        #ax = axes[plane_idx]    
        segments, segment_tri_indices, segment_edges = plane_mesh_slice(
                vertices, triangles, plane_origin=centroid, plane_normal=plane_normal, epsilon=0.0)
        tri_to_seg = {tri: i for i, tri in enumerate(segment_tri_indices)}
        groups = list(nx.connected_components(tri_adj_graph.subgraph(segment_tri_indices)))

        for group in groups:
            if len(group) <= 3:
                continue
            if len(np.intersect1d(list(group), centroid_tris) ) == 0:
                continue
            _, seg_indices = get_segment_path_from_triangle_path(tri_adj_graph, list(group), tri_to_seg)
            if seg_indices is None:
                continue
            lp = (segments[seg_indices][:, 0] + segments[seg_indices][:, 1]) / 2
            
            lp_vals, lp_vecs = np.linalg.eigh(np.cov(lp, rowvar=False) + np.outer(lp.mean(axis=0)-centroid, lp.mean(axis=0)-centroid))
            lp2 = (lp - centroid) @ lp_vecs[:, 1:]
            spline = CubicSpline(np.arange(len(lp2)), lp2)
            spline_vals = spline(np.arange(len(lp2)))
            spline_ders = spline(np.arange(len(lp2)), 1)
            tangents = spline_ders / np.linalg.norm(spline_ders, axis=1, keepdims=True)
            closest_idx_to_centroid = np.argmin(np.linalg.norm(lp - centroid, axis=1))
            comp1 = lp2 @ tangents[closest_idx_to_centroid,:] 
            if not (comp1.min() < 0 and comp1.max() > 0):
                continue
            comp2 = lp2 @ np.array([-tangents[closest_idx_to_centroid, 1], tangents[closest_idx_to_centroid, 0]])
            comp2_min, comp2_max = comp2.min(), comp2.max()
            if abs(comp2_min) > abs(comp2_max):
                comp2 = -comp2
            
            #x_s, s = arclength_resample(lp2, N=200)
            #linear_s = x_s[0] + np.outer(s, x_s[-1] - x_s[0])
            #residual_s = x_s - linear_s
            #coeffs = dst(residual_s, type=2, axis=0, norm='ortho')
            #coeffs[10:] = 0
            #x_curvature = curvature_dst(coeffs, x_s[0], x_s[-1], s)
            ax.plot(comp1, comp2)
            
            fig.add_trace(go.Scatter3d(
                x=lp[:, 0],
                y=lp[:, 1],
                z=lp[:, 2],
                mode='lines',
                marker=dict(size=2)
            ))
    ax.set_aspect('equal')
    plt.show()
    fig.show()
    


    continue 